# 🏦 Pandas for auditors — Solutions Level 2: Intermediate

**Context**: Review of the residential real-estate loan portfolio.

> ⚠️ This file contains the **solutions**. Try first with `exercise_intermediate.ipynb`!

## 0. Data generation

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
np.random.seed(2025)

n = 350

branches = ['North Branch', 'South Branch', 'East Branch', 'West Branch', 'Central Branch']
loan_types = ['Residential Real Estate', 'Commercial Real Estate', 'Buy-to-Let']
statuses = ['Paid', 'Minor Delay', 'Severe Delay', 'Unpaid']

payment_status = np.random.choice(statuses, n, p=[0.68, 0.15, 0.10, 0.07])
due_amounts = np.round(np.random.uniform(500, 4500, n), 2)
payment_ratio = np.where(
    payment_status == 'Paid', 1.0,
    np.where(payment_status == 'Minor Delay',
             np.random.uniform(0.0, 1.0, n),
             np.where(payment_status == 'Severe Delay',
                      np.random.uniform(0.0, 0.5, n),
                      0.0))
)
paid_amounts = np.round(due_amounts * payment_ratio, 2)

days_late = np.where(
    payment_status == 'Paid', 0,
    np.where(payment_status == 'Minor Delay', np.random.randint(1, 30, n),
    np.where(payment_status == 'Severe Delay', np.random.randint(30, 90, n),
             np.random.randint(90, 365, n)))
)

loans = pd.DataFrame({
    'loan_id':          [f'PR{str(i).zfill(5)}' for i in range(1, n + 1)],
    'client_id':        np.random.randint(50000, 51000, n),
    'branch':           np.random.choice(branches, n),
    'loan_type':        np.random.choice(loan_types, n, p=[0.55, 0.25, 0.20]),
    'interest_rate':    np.round(np.random.uniform(1.5, 4.5, n), 2),
    'due_date':         pd.to_datetime('2024-01-01') + pd.to_timedelta(
                            np.random.randint(0, 180, n), unit='D'),
    'due_amount':       due_amounts,
    'paid_amount':      paid_amounts,
    'payment_status':   payment_status,
    'days_overdue':     days_late,
})

loans.loc[np.random.choice(loans.index, 10, replace=False), 'interest_rate'] = np.nan
loans.loc[np.random.choice(loans.index, 5,  replace=False), 'paid_amount'] = np.nan

dups = loans.sample(4, random_state=99).copy()
dups['loan_id'] = [f'DUP{i}' for i in range(4)]
loans = pd.concat([loans, dups], ignore_index=True)
loans = loans.sample(frac=1, random_state=3).reset_index(drop=True)

print('Loans dataset ready:', loans.shape[0], 'installments,', loans.shape[1], 'columns')
loans.head()

---
## Exercise 1 — Multi-condition filtering

In [ ]:
# 1. Severe Delay or Unpaid
problematic = loans[loans['payment_status'].isin(['Severe Delay', 'Unpaid'])]
print('Severe delays + Unpaid:', len(problematic))
problematic.head()

In [ ]:
# 2. Unpaid with amount > 3,000
loans[(loans['payment_status'] == 'Unpaid') & (loans['due_amount'] > 3000)][
    ['loan_id', 'branch', 'loan_type', 'due_amount', 'days_overdue']
]

In [ ]:
# 3. Delay between 30 and 90 days
mid_delay = loans[loans['days_overdue'].between(30, 90)]
print('Installments with delay 30-90 days:', len(mid_delay))
mid_delay[['loan_id', 'branch', 'payment_status', 'days_overdue', 'due_amount']].head()

In [ ]:
# 4. Commercial/Buy-to-Let loans NOT paid
target_types = ['Commercial Real Estate', 'Buy-to-Let']
unpaid_pro = loans[
    loans['loan_type'].isin(target_types)
    & ~(loans['payment_status'] == 'Paid')
]
print('Commercial/Buy-to-Let loans not paid:', len(unpaid_pro))
unpaid_pro[['loan_id', 'branch', 'loan_type', 'payment_status', 'due_amount']].head()

---
## Exercise 2 — Computed columns

In [ ]:
# 1. Unpaid amount
loans['unpaid_amount'] = (loans['due_amount'] - loans['paid_amount'].fillna(0)).round(2)
loans[['due_amount', 'paid_amount', 'unpaid_amount']].head(8)

In [ ]:
# 2. Critical delay (boolean)
loans['critical_delay'] = loans['days_overdue'] >= 60
print('Critical delays (>= 60d):', loans['critical_delay'].sum())

In [ ]:
# 3. Delay category with np.select
conditions = [
    loans['days_overdue'] == 0,
    loans['days_overdue'].between(1, 59),
]
choices = ['None', 'Minor']
loans['delay_category'] = np.select(conditions, choices, default='Critical')
loans['delay_category'].value_counts()

---
## Exercise 3 — Summaries with `groupby`

In [ ]:
# 1. Summary by branch
branch_summary = loans.groupby('branch').agg(
    total_due=('due_amount', 'sum'),
    total_paid=('paid_amount', 'sum'),
    total_unpaid=('unpaid_amount', 'sum'),
).round(2)
branch_summary.sort_values('total_unpaid', ascending=False)

In [ ]:
# 2. Summary by loan type
loans.groupby('loan_type').agg(
    nb_installments=('loan_id', 'count'),
    avg_delay_days=('days_overdue', 'mean'),
    total_unpaid=('unpaid_amount', 'sum'),
).round(2)

In [ ]:
# 3. Number of installments by branch and status
loans.groupby(['branch', 'payment_status'])['loan_id'].count().unstack(fill_value=0)

---
## Exercise 4 — Pivot table (`pivot_table`)

In [ ]:
# 1. Pivot: unpaid amount by branch × loan type
pd.pivot_table(
    loans,
    index='branch',
    columns='loan_type',
    values='unpaid_amount',
    aggfunc='sum',
    fill_value=0,
).round(2)

In [ ]:
# 2. Pivot: number of installments by branch × status
pd.pivot_table(
    loans,
    index='branch',
    columns='payment_status',
    values='loan_id',
    aggfunc='count',
    fill_value=0,
)

---
## Exercise 5 — Data quality

In [ ]:
# 1. Missing values per column
loans.isna().sum()

In [ ]:
# 2. Business duplicates
business_keys = ['client_id', 'due_date', 'due_amount']
duplicates = loans[loans.duplicated(subset=business_keys, keep=False)]
print('Rows involved in a duplicate:', len(duplicates))
duplicates.sort_values(business_keys)[['loan_id', 'client_id', 'due_date', 'due_amount']].head(10)

In [ ]:
# 3. Cleaned DataFrame
loans_clean = (
    loans
    .drop_duplicates(subset=business_keys, keep='first')
    .copy()
)
loans_clean['paid_amount'] = loans_clean['paid_amount'].fillna(0)
print('Rows before cleaning:', len(loans))
print('Rows after cleaning:', len(loans_clean))

---
## Exercise 6 — Summary analysis: branches at risk

In [ ]:
# Work on the cleaned DataFrame
total_by_branch = loans_clean.groupby('branch').agg(
    total_count=('loan_id', 'count'),
    total_unpaid=('unpaid_amount', 'sum'),
)

# Number of severe delays + unpaid
defaults = loans_clean[loans_clean['payment_status'].isin(['Severe Delay', 'Unpaid'])]
default_count = defaults.groupby('branch')['loan_id'].count().rename('default_count')

# Merge and rate calculation
risk = total_by_branch.join(default_count, how='left').fillna(0)
risk['default_rate_pct'] = (risk['default_count'] / risk['total_count'] * 100).round(1)
risk['total_unpaid'] = risk['total_unpaid'].round(2)

risk.sort_values('default_rate_pct', ascending=False)